# **Explore FIR Filters with Audio** 

*(c) 2023 Barry Van Veen, AllSignalProcessing, LLC*

This notebook explores the effect of two different finite impulse response (FIR) filters on an audio signal.  The output of an FIR filter has a limited duration after the input ceases.  

(In contrast, the output of an infinite impulse response filter never goes exactly to zero once the input to the filter ceases.)

The two FIR filters considered are

1.   an averaging filter where output is the average of the eight most recent input samples
2.   a differencing filter where the output is the average difference of the two most recent inputs

If we let $x[n]$ be the input at time $n$ and $y[n]$ the output, then we may write the output in terms of the input as

Averaging:
$$ y[n] = \frac{1}{8}\left(x[n] + x[n-1] +x[n-2] + \cdots +x[n-7]\right)$$
Differencing:
$$y[n] = \frac{1}{2}\left(x[n]-x[n-1]\right)$$


Contrast the effect of these two filters in the plots and the audio.

In [ ]:
# Set up workspace

import numpy as np
import scipy.signal as signal
import matplotlib.pyplot as plt 

from scipy.io import loadmat
from google.colab import drive
drive.mount('/content/drive')

from IPython.display import Audio, display

# set default figure size and resolution
plt.rcParams['figure.figsize'] = [8, 6]
plt.rcParams['figure.dpi'] = 100

# load MATLAB file containing song data
# may need to change directory address depending on user configuration

path = '/content/drive/My Drive/Colab Notebooks/ASP/Foundations/StealthGroover_sub4.mat'

in_data = loadmat(path)

# "Stealth Groover" Kevin MacLeod (incompetech.com) 
# Licensed under Creative Commons: By Attribution 3.0
# http://creativecommons.org/licenses/by/3.0/
# 
# Subsampled by a factor of 4 to obtain a sampling rate of 12kHz

fs = int(in_data['fs'])
s = in_data['Sl']

# Note: Audio expects channels x time values arrangment so the number of rows is maximum of 2
song = np.transpose(s)

# define the indices and time interval used when plotting a section of the song
timeindex = np.arange(52000,53000)
time = timeindex/fs

# filter coefficients for moving average of eight most recent values
b_av = np.array([.125, .125, .125, .125, .125, .125, .125, .125])

a = np.array([1]);

# filter coefficients for difference of most recent values
b_dif = np.array([0.5, -0.5])

# step signal for warm up activity
stepsignal = np.concatenate(( np.zeros((30,1)), 2*np.ones((40,1)), np.ones((30,1)) ))


# Warm Up

Here we first consider a simple example to begin developing intuition about the effects of the averaging and differencing filters.  Both filters are applied to a signal that has sudden transitions between constant values.

In [ ]:
# Apply averaging filter to step signal
av_filt_step = signal.lfilter(b_av,a,stepsignal, axis = 0)

plt.stem(stepsignal, markerfmt='o',use_line_collection=True)
plt.stem(av_filt_step, markerfmt='x',use_line_collection=True)
plt.xlabel('Sample')
plt.ylabel('Amplitude')
plt.legend(['input','output'])
plt.title('Average Filtered Step Signal')
plt.show();

# Apply differencing filter to step signal
dif_filt_step = signal.lfilter(b_dif,a,stepsignal, axis = 0)

plt.stem(stepsignal, markerfmt='o',use_line_collection=True)
plt.stem(dif_filt_step, markerfmt='x',use_line_collection=True)
plt.xlabel('Sample')
plt.ylabel('Amplitude')
plt.legend(['input','output'])
plt.title('Difference Filtered Step Signal')
plt.show();

Explain why the output of each filter looks the way it does. What are the implications of this behavior?


# Original Sound Clip

In [ ]:
# plot small portion of original clip

plt.plot(time,song[0,timeindex])
plt.xlabel('Time (sec)')
plt.ylabel('Amplitude')
plt.title('Section of Original Song')
plt.show()

The graph shows 1000 samples of the song, which correspond to about 1/12 second total time duration.  We are not using a stem plot here because it would be hard to see the variations in the signal with the number of values being shown. We see that there are components of the amplitude that vary rapidly as well as components that vary more slowly over the time interval shown.  We are going to pay attention to how both of those change when the data is filtered.

### Audio Object for song

In [ ]:
# Create audio object

Audio(song, rate=fs)


Click on the play button to listen to eight seconds of the song (much longer than the segment shown in the previous graph) paying attention to the high frequency sounds and the low frequency sounds.

# Apply averaging FIR filter to song clip

This filter forms the output at each time instant as the average of the eight most recent input values.

How do you expect this averaging process to change the character of the amplitude? (Note the previous graph showed 1000 samples, so the averaging used for each output value is occuring over less than 1 percent of the waveform depicted.)

How do you predict the sound associated with the filtered song will differ from the original?

In [ ]:
# apply the averaging filter defined by b_av and a to the song 
filt_song_av = signal.lfilter(b_av,a,song)

plt.plot(time,song[0,timeindex],'b',time,filt_song_av[0,timeindex],'r')
plt.xlabel('Time (sec)')
plt.ylabel('Amplitude')
plt.title('Section of Average-Filtered Song')
plt.legend(['Original', 'Average Filtered'])
plt.show()



Compare the graph of the average filtered song to the original.  What is the effect of averaging the most recent values?



### Audio Object for Average-Filtered Song


In [ ]:
# Create audio object

Audio(filt_song_av, rate=fs)

How does the character of the song change relative to the original audio clip?


Is there a greater change in the high frequencies or the low frequencies?  Is this change consistent with what you see in the graphs?

# Apply differencing FIR filter to song clip

This filter forms the output at each time instant as the most recent minus the previouse input value divided by two.

How do you expect this differencing process to change the character of the amplitude? To get you started, think about the output of the differencing process if the input is approximately constant for multiple samples in a row.

How do you predict the sound associated with the filtered song will differ from the original?

In [ ]:
# apply the differencing filter defined by b_dif and a to the song 
filt_song_dif = signal.lfilter(b_dif,a,song)

plt.plot(time,song[0,timeindex],'b',time,filt_song_dif[0,timeindex],'r')
plt.xlabel('Time (sec)')
plt.ylabel('Amplitude')
plt.title('Section of Difference-Filtered Song')
plt.legend(['Original', 'Difference Filtered'])
plt.show()


Compare the graph of the difference filtered song to that of the original and the average filtered song?  What is the effect of taking the difference of two successive values? 

Contrast the features of the difference filtered song with those of the average filtered song.

### Audio Object for Difference-Filtered Song

In [ ]:
# Create Audio Object

Audio(filt_song_dif, rate=fs)

How does the character of the difference-filtered song change relative to that of the original audio clip?

Is there a greater change in the high frequencies or the low frequencies?  Is this change consistent with what you see in the graphs?

Contrast the character of the difference-filtered song and the average-filtered song both graphically and in terms of the audio.

# Explore Further

You can change the way past values are combined by modifying the numpy array `b_av` or `b_dif`.  For example to average only the two most recent inputs and compute 
$$ y[n] = \frac{1}{2}\left(x[n] + x[n-1]\right) $$
we replace


```
b_av = np.array([.125, .125, .125, .125, .125, .125, .125, .125])
```

with


```
b_av = np.array([0.5, 0.5])
``` 

### Ideas
1.   Compare the effects of averaging different numbers of past values, such as two, eight, sixteen, etc.  How does averaging more past values affect the filter output?
2.   Try variations on differencing, such as `b_dif = np.array([0.5, 0, -0.5])` and see if you can explain the characteristics of the filter output.  A tip is to consider what input values result in zero output.  For example, the case suggested will produce exactly zero output whenever a section of the input is constant ($x[n] = \left\{ \ldots, 1, 1, 1, 1, \ldots \right\}$ OR alternates sign ($x[n] = \left\{ \ldots, 1, -1, 1, -1, \ldots \right\}$.

In the future we learn how systematic methods to characterize the effect of filter weights and to design weights that impart specified characteristics to the filter output.



